In [1]:
# Чекпойнт 6: Нейросетевые архитектуры для классификации ОКПД2

import kagglehub
import numpy as np
import pandas as pd
import pickle
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
import warnings
import re
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU доступен: {tf.config.list_physical_devices('GPU')}")
warnings.filterwarnings('ignore')

2026-06-11 07:33:48.560408: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781163228.804490      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781163228.869503      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781163229.445939      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781163229.445996      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781163229.445999      58 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU доступен: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# 1. ЗАГРУЗКА ДАННЫХ

path = kagglehub.dataset_download("aldarovalexander/checkpoint6-data")
print(f"Path to dataset files: {path}")

with open(f"{path}/checkpoint6_data.pkl", 'rb') as f:
    data = pickle.load(f)

X_train_bow = data['X_train_bow']
X_val_bow = data['X_val_bow']
X_test_bow = data['X_test_bow']
y_train_bin = data['y_train_bin']
y_val_bin = data['y_val_bin']
y_test_bin = data['y_test_bin']
mlb = data['mlb']
classes = data['classes']

print(f"X_train_bow shape: {X_train_bow.shape}")
print(f"X_val_bow shape: {X_val_bow.shape}")
print(f"X_test_bow shape: {X_test_bow.shape}")
print(f"Количество классов: {len(classes)}")

print("\nКонвертация в плотные массивы")
X_train_dense = X_train_bow.toarray().astype(np.float32)
X_val_dense = X_val_bow.toarray().astype(np.float32)
X_test_dense = X_test_bow.toarray().astype(np.float32)

print(f"X_train_dense shape: {X_train_dense.shape}")
print(f"Память: {X_train_dense.nbytes / 1024**2:.1f} MB")

Path to dataset files: /kaggle/input/datasets/aldarovalexander/checkpoint6-data
X_train_bow shape: (138362, 8000)
X_val_bow shape: (29649, 8000)
X_test_bow shape: (29649, 8000)
Количество классов: 84

Конвертация в плотные массивы
X_train_dense shape: (138362, 8000)
Память: 4222.5 MB


In [3]:
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ

def create_model(input_dim, num_classes, architecture='deep', dropout_rate=0.5):

    model = keras.Sequential()
    
    if architecture == 'shallow':
        model.add(layers.Dense(256, activation='relu', input_shape=(input_dim,)))
        model.add(layers.Dropout(dropout_rate))
        
    elif architecture == 'deep':
        model.add(layers.Dense(512, activation='relu', input_shape=(input_dim,)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(256, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        
    elif architecture == 'wider':
        model.add(layers.Dense(1024, activation='relu', input_shape=(input_dim,)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.Dense(512, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
        
    elif architecture == 'dropout_heavy':
        model.add(layers.Dense(512, activation='relu', input_shape=(input_dim,)))
        model.add(layers.Dropout(0.6))
        model.add(layers.Dense(256, activation='relu'))
        model.add(layers.Dropout(0.6))
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dropout(0.5))
    
    model.add(layers.Dense(num_classes, activation='sigmoid'))
    
    return model

def train_and_evaluate(model, X_train, y_train, X_val, y_val, 
                       epochs=30, batch_size=64, learning_rate=0.001,
                       model_name="Model"):

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    early_stop = callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )

    start_time = time.time()
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    train_time = time.time() - start_time

    y_pred = model.predict(X_val)
    y_pred_binary = (y_pred > 0.5).astype(int)
    accuracy = accuracy_score(y_val, y_pred_binary)
    
    print(f"\n{model_name} - Результаты:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Время обучения: {train_time:.1f} сек ({train_time/60:.1f} мин)")
    
    return model, history, accuracy, train_time

In [4]:
# 1. ЗАГРУЗКА СЫРОГО ДАТАСЕТА

from kagglehub import KaggleDatasetAdapter

file_path = "contracts_dataset_unique.json"

dataset_path = kagglehub.dataset_download("aldarovalexander/contract")
full_file_path = f"{dataset_path}/{file_path}"

df = pd.read_json(full_file_path, dtype={'regNum': str})

print(f"Загружено {len(df)} записей")
print(f"Колонки: {df.columns.tolist()}")

print("\nПример сырого текста ДО обработки:")
print(df['contractSubjectFull'].iloc[0][:500])
print("...")

Загружено 199913 записей
Колонки: ['regNum', 'contractSubjectFull', 'OKPD2_codes']

Пример сырого текста ДО обработки:
1.1. Подрядчик обязуется выполнить работы по ремонту кровли и утеплению труб жилого дома №6, расположенного по адресу: Тульская область, Ленинский район, п. Молодежный, ул. Центральная (далее - объект) в соответствии с условиями настоящего контракта и локальной сметой (приложение №1), являющейся неотъемлемой частью настоящего контракта. 1.2. Подрядчик обязуется выполнить работы, указанные в пункте 1.1. контракта, своими силами или с привлечением субподрядных организаций. 2.
...


In [5]:
# 3. МИНИМАЛЬНАЯ ОЧИСТКА ТЕКСТОВ И ПОДГОТОВКА ДАННЫХ (ВСЕ КЛАССЫ)

# Используем ВСЕ уникальные коды
all_codes = []
for codes in df['OKPD2_codes']:
    if codes:
        all_codes.extend(codes)

unique_codes = sorted(set(all_codes))
print(f"Всего уникальных кодов: {len(unique_codes)}")

def has_codes(codes):
    return codes is not None and len(codes) > 0

mask = df['OKPD2_codes'].apply(has_codes)
df_filtered = df[mask].copy()
df_filtered['target'] = df_filtered['OKPD2_codes'].apply(lambda x: x)

print(f"Размер выборки: {len(df_filtered)}")

mlb = MultiLabelBinarizer(classes=unique_codes)
y_bin = mlb.fit_transform(df_filtered['target'])
print(f"Количество классов: {len(mlb.classes_)}")

def minimal_clean_text(text):

    # Минимальная очистка для RNN.
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\b\w{1,2}\b', ' ', text)

    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

texts = df_filtered['contractSubjectFull'].apply(minimal_clean_text).values

print(f"\nПример текста ПОСЛЕ минимальной очистки:")
print(texts[0][:500])
print("...")

text_lengths = [len(t) for t in texts]
print(f"\nСтатистика по длинам текстов:")
print(f"  Средняя длина: {np.mean(text_lengths):.0f} символов")
print(f"  Медианная длина: {np.median(text_lengths):.0f} символов")
print(f"  Максимальная длина: {np.max(text_lengths):.0f} символов")
print(f"  Минимальная длина: {np.min(text_lengths):.0f} символов")

for p in [50, 75, 90, 95, 99]:
    print(f"  {p}% перцентиль: {np.percentile(text_lengths, p):.0f} символов")

Всего уникальных кодов: 84
Размер выборки: 199913
Количество классов: 84

Пример текста ПОСЛЕ минимальной очистки:
. . подрядчик обязуется выполнить работы ремонту кровли утеплению труб жилого дома № , расположенного адресу: тульская область, ленинский район, . молодежный, . центральная (далее - объект) соответствии условиями настоящего контракта локальной сметой (приложение № ), являющейся неотъемлемой частью настоящего контракта. . . подрядчик обязуется выполнить работы, указанные пункте . . контракта, своими силами или привлечением субподрядных организаций. .
...

Статистика по длинам текстов:
  Средняя длина: 814 символов
  Медианная длина: 640 символов
  Максимальная длина: 8789 символов
  Минимальная длина: 1 символов
  50% перцентиль: 640 символов
  75% перцентиль: 976 символов
  90% перцентиль: 1521 символов
  95% перцентиль: 1893 символов
  99% перцентиль: 3203 символов


In [6]:
# 4. РАЗДЕЛЕНИЕ НА ВЫБОРКИ


from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, y_bin, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {len(X_train)} ({len(X_train)/len(texts)*100:.1f}%)")
print(f"Val: {len(X_val)} ({len(X_val)/len(texts)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(texts)*100:.1f}%)")
print(f"Количество классов: {y_bin.shape[1]}")

Train: 139939 (70.0%)
Val: 29987 (15.0%)
Test: 29987 (15.0%)
Количество классов: 84


In [7]:
# 5. ТОКЕНИЗАЦИЯ ДЛЯ RNN

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_FEATURES = 100000  # Размер словаря
MAX_LEN = 3000         # Максимальная длина последовательности 

tokenizer = Tokenizer(num_words=MAX_FEATURES, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"X_val_pad shape: {X_val_pad.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")
print(f"Размер словаря: {len(tokenizer.word_index)}")
print(f"Уникальных токенов в обучении: {len(set(tokenizer.word_index.keys()))}")

X_train_pad shape: (139939, 3000)
X_val_pad shape: (29987, 3000)
X_test_pad shape: (29987, 3000)
Размер словаря: 132844
Уникальных токенов в обучении: 132844


In [8]:
# 7. МОДЕЛЬ 2: BIDIRECTIONAL GRU (исправленная)

num_classes = y_bin.shape[1]
print(f"Количество классов: {num_classes}")

def create_gru_model(vocab_size=MAX_FEATURES, embedding_dim=256, max_len=MAX_LEN, num_classes=num_classes):
    model = tf.keras.Sequential([
        layers.Embedding(vocab_size, embedding_dim, input_length=max_len),
        layers.Bidirectional(layers.GRU(128, return_sequences=True, dropout=0.3)),
        layers.Bidirectional(layers.GRU(64, dropout=0.3)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='sigmoid')
    ])
    return model

gru_model = create_gru_model()
gru_model.build(input_shape=(None, MAX_LEN))
gru_model.summary()

gru_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

print("\nНачало обучения GRU...")
start_time = time.time()
gru_history = gru_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
gru_time = time.time() - start_time

y_pred_gru = gru_model.predict(X_test_pad)
y_pred_gru_bin = (y_pred_gru > 0.5).astype(int)
gru_accuracy = accuracy_score(y_test, y_pred_gru_bin)

print(f"\nGRU - Результаты:")
print(f"  Accuracy: {gru_accuracy:.4f}")
print(f"  Время обучения: {gru_time:.1f} сек ({gru_time/60:.1f} мин)")

Количество классов: 84


I0000 00:00:1781163350.546280      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781163350.552335      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 3000, 256)      │    25,600,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 3000, 256)      │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 84)             │         5,460 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,050,324 (99.37 MB)

 Trainable params: 26,050,324 (99.37 MB)

 Non-trainable params: 0 (0.00 B)


Начало обучения GRU...
Epoch 1/10


I0000 00:00:1781163360.349301     131 cuda_dnn.cc:529] Loaded cuDNN version 91002


1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 958ms/step - accuracy: 0.4547 - loss: 0.0466 - val_accuracy: 0.6652 - val_loss: 0.0221 - learning_rate: 0.0010
Epoch 2/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1055s 964ms/step - accuracy: 0.7090 - loss: 0.0208 - val_accuracy: 0.7682 - val_loss: 0.0162 - learning_rate: 0.0010
Epoch 3/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 966ms/step - accuracy: 0.7859 - loss: 0.0159 - val_accuracy: 0.7892 - val_loss: 0.0149 - learning_rate: 0.0010
Epoch 4/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 965ms/step - accuracy: 0.8185 - loss: 0.0135 - val_accuracy: 0.7984 - val_loss: 0.0143 - learning_rate: 0.0010
Epoch 5/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 965ms/step - accuracy: 0.8387 - loss: 0.0120 - val_accuracy: 0.8039 - val_loss: 0.0142 - learning_rate: 0.0010
Epoch 6/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 966ms/step - accuracy: 0.8531 - loss: 0.0109 - val_accuracy: 0.8063 - val_loss: 0.0145 - learning_rate: 0.0010
Epoch 7/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1056s 965ms/ste

In [11]:
from tensorflow.keras.models import save_model
import joblib
import pickle
from pathlib import Path

MODEL_DIR = Path("/kaggle/working/saved_models")
MODEL_DIR.mkdir(exist_ok=True)

# Сохраняем модель
save_model(gru_model, MODEL_DIR / "bigru_prd.keras")
print("✅ Модель сохранена")

# Сохраняем токенизатор
joblib.dump(tokenizer, MODEL_DIR / "bigru_tokenizмайзнйer.pkl")
print("✅ Токенизатор сохранён")

# Сохраняем классы
with open(MODEL_DIR / "classes.pkl", 'wb') as f:
    pickle.dump(classes, f)
print("✅ Классы сохранены")

# Сохраняем конфиг
import json
config = {
    'max_features': 100000,
    'max_len': 3000,
    'num_classes': len(classes),
    'embedding_dim': 256,
    'gru_units': [128, 64],
    'dropout': 0.3,
    'test_accuracy': gru_accuracy  # из ячейки 7
}
with open(MODEL_DIR / "bigru_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Всё сохранено в {MODEL_DIR}")

✅ Модель сохранена
✅ Токенизатор сохранён
✅ Классы сохранены

✅ Всё сохранено в /kaggle/working/saved_models


In [12]:
from IPython.display import FileLink
from pathlib import Path

MODEL_DIR = Path("/kaggle/working/saved_models")

print("📁 Нажмите на ссылки для скачивания:\n")
for f in MODEL_DIR.iterdir():
    print(f"   {FileLink(str(f))}")

📁 Нажмите на ссылки для скачивания:

   /kaggle/working/saved_models/classes.pkl
   /kaggle/working/saved_models/bigru_config.json
   /kaggle/working/saved_models/bigru_tokenizer.pkl
   /kaggle/working/saved_models/bigru_prd.keras
